In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Vertex AI Model Garden - OSS Distillation Feasibility Study

<table><tbody><tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/notebooks/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/vertex-ai-samples/main/notebooks/community/model_garden/model_garden_oss_distillation_feasibility.ipynb">
      <img alt="Workbench logo" src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" width="32px"><br> Run in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fvertex-ai-samples%2Fmain%2Fnotebooks%2Fcommunity%2Fmodel_garden%2Fmodel_garden_oss_distillation_feasibility.ipynb">
      <img alt="Google Cloud Colab Enterprise logo" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" width="32px"><br> Run in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/model_garden/model_garden_oss_distillation_feasibility.ipynb">
      <img alt="GitHub logo" src="https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png" width="32px"><br> View on GitHub
    </a>
  </td>
</tr></tbody></table>

## Overview
This notebook serves as a critical validation step in the Model Distillation pipeline. Before committing resources to distill a smaller "Student" model to mimic a larger "Teacher" model, we must first determine if the model pair and the target dataset are compatible.

### Core Objective
The primary goal is to calculate and compare the Perplexity scores of both the Teacher and Student models on a specific domain dataset, in this case, [syz-ml2025/medmcqa](https://huggingface.co/datasets/syz-ml2025/medmcqa) (Medical Multiple-Choice QA).

### Why Perplexity Matters
Perplexity is a measurement of how well a probability model predicts a sample.

- **Teacher Validation:** We use the `deepseek-ai/deepseek-r1-0528-maas` model via the [Vertex AI MaaS API](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/call-open-model-apis). A low teacher perplexity confirms that the teacher understands the domain well enough to provide high-quality supervision.

- **Gap Identification:** We evaluate the `Qwen/Qwen3-0.6B` student model using the HuggingFace library. By quantifying the performance gap between the teacher and the student, we can estimate the potential "knowledge headroom" available for distillation.

### Workflow Steps
Environment Setup: Installation of prerequisite libraries (Vertex AI SDK, Transformers, Datasets).

- **Dataset Sampling:** Extraction of a representative subset (200 samples) from the medmcqa training split to ensure efficient benchmarking.

- **Teacher Inference:** Leveraging Vertex AI Model-as-a-Service (MaaS) to compute log-likelihoods and derive the teacher's perplexity.

- **Student Inference:** Running the student model on local GPU resources (NVIDIA L4) to calculate its baseline perplexity before any training occurs.

### Success Criteria
A successful prerequisite check is identified when the Teacher model shows significantly lower perplexity than the Student, suggesting that the Student has a meaningful opportunity to learn the Teacher's superior reasoning and linguistic patterns within the medical context.

## Before you begin

In [ ]:
# @markdown Install the prerequisite libraries.
!pip3 install --upgrade datasets transformers

In [ ]:
import json

# @markdown Create a small test dataset using HF dataset for perplexity calculation.
from datasets import load_dataset


def create_test_dataset(sample_dataset, test_file_path, dataset_split, num_samples=200):
    dataset = load_dataset(sample_dataset, split=dataset_split)
    test_samples = dataset.select(range(num_samples))
    with open(test_file_path, "w") as f:
        for sample in test_samples:
            f.write(json.dumps(sample) + "\n")


def format_dataset(dataset):
    """Formats the dataset for perplexity calculation."""
    formatted_data = []
    with open(dataset, "r") as f:
        data = [json.loads(line) for line in f if line.strip()]
        for record in tqdm.tqdm(data):
            cop_map = {0: "A", 1: "B", 2: "C", 3: "D"}
            text = (
                f'{record["question"]}\nA) {record["opa"]}\nB) {record["opb"]}\nC) '
                f'{record["opc"]}\nD) {record["opd"]}'
            )
            if record["exp"]:
                text += f'\n{record["exp"]}'
            text += f'\nThe answer is {cop_map[record["cop"]]}'
            formatted_data.append(text)

    return formatted_data


sample_dataset = "syz-ml2025/medmcqa"  # @param {type:"string"}
test_file_path = "test.jsonl"  # @param {type:"string"}
dataset_split = "train"  # @param {type:"string"}
dataset_sample_size = 200  # @param {type:"integer"}

create_test_dataset(
    sample_dataset=sample_dataset,
    test_file_path=test_file_path,
    dataset_split=dataset_split,
    num_samples=dataset_sample_size,
)
formatted_dataset = format_dataset(test_file_path)

In [ ]:
# @markdown Perplexity calculation on the test dataset using VMG MaaS API for the teacher model.

import concurrent.futures
import os
import subprocess
import sys
from typing import Any, Tuple

import openai
import tenacity


def get_access_token() -> str:
    """Gets the access token from gcloud."""
    try:
        return subprocess.check_output(
            ["gcloud", "auth", "print-access-token"], encoding="utf-8"
        ).strip()
    except subprocess.CalledProcessError as e:
        print("Error getting access token: %r", e)
        sys.exit(1)


@tenacity.retry(
    stop=tenacity.stop_after_attempt(10),
    wait=tenacity.wait_exponential(multiplier=1, min=2, max=60),
    retry=tenacity.retry_if_exception_type(
        (
            openai.APIConnectionError,
            openai.RateLimitError,
            openai.APIStatusError,
        )
    ),
)
def _create_completion_with_retry(
    client: openai.OpenAI,
    messages: list[dict[str, str]],
    model: str,
) -> Any:
    """Creates a completion with retry logic to get logprobs."""
    return client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=16384,
        logprobs=True,
        extra_body={"prompt_logprobs": 0},
        temperature=0.0,
    )


def calculate_perplexity_item(
    messages,
    client: openai.OpenAI,
    model: str,
) -> Tuple[float, int]:
    """Calculates sum neg log likelihood and token count for a single record."""
    response = _create_completion_with_retry(
        client=client,
        messages=messages,
        model=model,
    )

    logprobs_list = []
    # Check for prompt_logprobs field first
    if hasattr(response, "prompt_logprobs") and response.prompt_logprobs:
        # prompt_logprobs is like [None, {'token1': {'logprob': -0.1}}, ...]
        for token_logprob in response.prompt_logprobs[1:]:
            if token_logprob:
                logprobs_list.append(list(token_logprob.values())[0]["logprob"])
    # If not, check logprobs in choices (OpenAI format for chat output tokens)
    # in case MaaS populates it with prompt logprobs
    elif (
        response.choices
        and response.choices[0].logprobs
        and response.choices[0].logprobs.content
    ):
        logprobs_list = [
            lp.logprob
            for lp in response.choices[0].logprobs.content
            if lp and lp.logprob is not None
        ]
    else:
        return 0.0, 0

    if not logprobs_list:
        return 0.0, 0

    sum_nll = -np.sum(logprobs_list)
    token_count = len(logprobs_list)
    return sum_nll, token_count


def calculate_perplexity_teacher(
    model: str,
    project_id: str,
    region: str,
    dataset: list[str],
    max_workers: int = 10,
) -> None:
    """Calculate the Perplexity score of the teacher model using VMG MaaS API.

    Args:
        model: Model ID for MaaS.
        project_id: GCP Project ID.
        region: GCP Region.
        dataset: The dataset.
        max_workers: Number of parallel workers.
    """
    api_key = get_access_token()
    if region == "global":
        api_endpoint = "aiplatform.googleapis.com"
    else:
        api_endpoint = f"{region}-aiplatform.googleapis.com"
    base_url = f"https://{api_endpoint}/v1/projects/{project_id}/locations/{region}/endpoints/openapi"
    print(f"Using MaaS base URL for the teacher model: {base_url}")

    client = openai.OpenAI(
        base_url=base_url,
        api_key=api_key,
    )
    per_sequence_ppls = []
    total_tokens = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        records = [[{"role": "user", "content": text}] for text in dataset]

        futures = [
            executor.submit(
                calculate_perplexity_item,
                record,
                client,
                model,
            )
            for record in records
        ]
        for future in tqdm.tqdm(
            concurrent.futures.as_completed(futures), total=len(futures)
        ):
            sum_nll, token_count = future.result()
            if token_count > 0:
                total_tokens += token_count
                per_sequence_ppls.append(np.exp(sum_nll / token_count))

    if total_tokens == 0:
        print("\n\nNo perplexity results calculated for the teacher model.")
        return

    ppl = np.mean(per_sequence_ppls)
    print(f"\n\nTeacher model Perplexity for the given dataset: {ppl:.4f}")


# Execute
teacher_maas_model_id = "deepseek-ai/deepseek-r1-0528-maas"  # @param {type:"string"}
teacher_project = os.environ["GOOGLE_CLOUD_PROJECT"]
teacher_region = "us-central1"  # @param {type:"string"}

calculate_perplexity_teacher(
    teacher_maas_model_id,
    teacher_project,
    teacher_region,
    formatted_dataset,
)

In [ ]:
# @markdown Perplexity calculation on the test dataset using HuggingFace for the student model. This section will require a GPU runtime environment to execute.

import numpy as np
import torch
import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


def calculate_perplexity_student(model_name, dataset, enable_thinking=True) -> None:
    """Runs perplexity calculation on input dataset for student model using HuggingFace.

    Args:
        model_name: HuggingFace model ID or local path.
        dataset: The dataset to calculate ppl on.
        enable_thinking: Whether to enable thinking in chat template.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
    model.eval()

    max_length = model.config.max_position_embeddings
    loss_fct = torch.nn.CrossEntropyLoss(reduction="none")

    per_sequence_ppls = []
    total_tokens = 0

    for text in tqdm.tqdm(dataset):
        encodings = tokenizer(text, return_tensors="pt")
        seq_len = encodings.input_ids.size(1)

        chunk_nlls = []
        for begin_loc in range(0, seq_len, max_length):
            end_loc = min(begin_loc + max_length, seq_len)
            input_ids = encodings.input_ids[:, begin_loc:end_loc].to(model.device)

            with torch.no_grad():
                outputs = model(input_ids)
                logits = outputs.logits

            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = input_ids[..., 1:].contiguous().to(logits.device)

            nll = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)
            )
            chunk_nlls.append(nll)

        all_nlls = torch.cat(chunk_nlls)
        if all_nlls.numel() > 0:
            sum_nll_seq = all_nlls.sum().item()
            token_count_seq = all_nlls.numel()
            per_sequence_ppls.append(np.exp(sum_nll_seq / token_count_seq))
            total_tokens += token_count_seq

    if total_tokens == 0:
        print("\n\nNo perplexity results calculated for the student model.")
        return

    ppl = np.mean(per_sequence_ppls)
    print(f"\n\nStudent model Perplexity for the given dataset: {ppl:.4f}")


# Execute
student_model_id = "Qwen/Qwen3-0.6B"  # @param {type:"string"}
enable_student_thinking = True  # @param {type:"boolean"}

calculate_perplexity_student(
    student_model_id, formatted_dataset, enable_student_thinking
)